# 02 — Exploring the output data

Load and explore the data products `laser-init` generates: the spatial
GeoPackage and the demographic CSVs.

Assumes a dataset for **ETH** exists (run `01_getting_started.ipynb`
first, or the cell below will generate it).

In [ ]:
import subprocess
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

COUNTRY, LEVEL, START, END = "ETH", 2, 2015, 2017
BASE = Path(COUNTRY) / str(START)
if not (BASE / "config.yaml").exists():
    subprocess.run(["laser-init", COUNTRY, str(LEVEL), str(START), str(END)], check=True)

## Spatial data: population and density

The GeoPackage is in EPSG:4326 (degrees), so we reproject to an equal-area CRS
(EPSG:6933) before computing area in km².

In [ ]:
gdf = gpd.read_file(BASE / f"{COUNTRY}_admin{LEVEL}.gpkg")
gdf["area_km2"] = gdf.to_crs(6933).geometry.area / 1e6
gdf["density"] = gdf.population / gdf.area_km2
gdf[["name", "population", "area_km2", "density"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
gdf.plot(column="population", ax=axes[0], legend=True, cmap="YlOrRd")
axes[0].set_title("Population"); axes[0].axis("off")
gdf.plot(column="density", ax=axes[1], legend=True, cmap="plasma")
axes[1].set_title("Density (per km^2)"); axes[1].axis("off")
plt.tight_layout()

## Demographic data

`age_dist.csv` (age structure), `cxr.csv` (crude birth/death rates by year), and
`life_exp.csv` (cumulative deaths for the survival curve).

In [ ]:
age = pd.read_csv(BASE / "age_dist.csv")
cxr = pd.read_csv(BASE / "cxr.csv")
life = pd.read_csv(BASE / "life_exp.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].barh(age.AgeGrpStart, age.PopTotal); axes[0].set_title("Age distribution")
axes[0].set_xlabel("population"); axes[0].set_ylabel("age group start")
axes[1].plot(cxr.Time, cxr.CBR, label="CBR"); axes[1].plot(cxr.Time, cxr.CDR, label="CDR")
axes[1].set_title("Birth/death rates"); axes[1].set_xlabel("year"); axes[1].legend()
axes[2].plot(life.cumulative_deaths); axes[2].set_title("Cumulative deaths (survival)")
axes[2].set_xlabel("age index")
plt.tight_layout()

These are exactly the inputs the generated model consumes for vital dynamics
(births by CBR) and mortality (the Kaplan–Meier survival curve).